# Script 20: Hierarchical KAN Architecture (Git Deployment)
This notebook clones the latest codebase from GitHub and launches the Multi-GPU training pipeline using HuggingFace Accelerate.

In [ ]:
# 1. Git Repository Configuration & Clone
import os
import shutil

# ── Cấu hình Git Repo của bạn ──
GIT_USERNAME = "JustinYuanZe"  # Thay đổi thành username GitHub của bạn
GIT_REPO = "SP1_TF_Binding_Project"      # Tên repository
GIT_TOKEN = ""             # Điền GitHub Personal Access Token nếu là Private Repo, để trống nếu Public

# Xây dựng URL clone
if GIT_TOKEN:
    REPO_URL = f"https://{GIT_TOKEN}@github.com/{GIT_USERNAME}/{GIT_REPO}.git"
else:
    REPO_URL = f"https://github.com/{GIT_USERNAME}/{GIT_REPO}.git"

# Dọn dẹp thư mục cũ nếu có để tránh xung đột
if os.path.exists(GIT_REPO):
    shutil.rmtree(GIT_REPO)
    print(f"🧹 Removed existing folder: {GIT_REPO}")

# Thực hiện clone repository từ GitHub
print(f"🚀 Cloning repository from {GIT_USERNAME}/{GIT_REPO}...")
exit_code = os.system(f"git clone {REPO_URL}")

if exit_code == 0:
    print("✅ Clone successful!")
    # Copy file script chính ra ngoài để chạy
    shutil.copy(f"{GIT_REPO}/notebooks/20_hierarchical_kan_kaggle.py", "./")
    print("✅ Copied 20_hierarchical_kan_kaggle.py to working directory!")
else:
    print("❌ ERROR: Git clone failed. Please check your username, repo name, or token (if private).")


In [ ]:
# 2. Launch the script using HuggingFace Accelerate
import subprocess
import torch

n_gpus = torch.cuda.device_count()
print(f'Detected {n_gpus} GPUs.')

cmd = ['accelerate', 'launch', '--mixed_precision=no']
if n_gpus > 1:
    cmd.extend(['--multi_gpu', f'--num_processes={n_gpus}'])
else:
    cmd.extend(['--num_processes=1'])

cmd.append('20_hierarchical_kan_kaggle.py')

print(f'Running command: {" ".join(cmd)}')
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True)
for line in process.stdout:
    print(line, end='')
process.wait()
print(f'Finished with exit code {process.returncode}')
